In [2]:
import pandas as pd
import numpy as np
import os
import pickle
from sklearn.ensemble import RandomForestRegressor

os.chdir('../')
%pwd

'/media/om/volume2/MLOPS/The Ultimate MLOPS Course/youtube_project/MLOPS_youtube_CTA_Project'

In [3]:
df= pd.read_csv("./data/raw/youtube_10000_videos.csv")
df.head()

,category,channel_id,channel_name,subscriber_count,channel_view_count,channel_video_count,video_id,video_title,published_at,duration_seconds,view_count,like_count,comment_count,description,video_url
0,Education,UCBg_mociSFb4ECbu91_cXJA,ABHYAAS EDUCATION,167000,5169057,1711,MOlaKBJ1nDA,प्रश्नावली 5.2 Class 10 Maths | NCERT Class 10...,2026-07-18T01:52:16Z,3988.0,427,21,3,📚 Class 10 Maths Chapter 5 – Arithmetic Progre...,https://www.youtube.com/watch?v=MOlaKBJ1nDA
1,Education,UCBg_mociSFb4ECbu91_cXJA,ABHYAAS EDUCATION,167000,5169057,1711,uXRH_uqCOXI,प्रश्नावली 5.3 Class 10 Maths | NCERT Class 10...,2026-07-19T02:22:39Z,3606.0,528,32,2,📚 Class 10 Maths Chapter 5 – Arithmetic Progre...,https://www.youtube.com/watch?v=uXRH_uqCOXI
2,Education,UCBg_mociSFb4ECbu91_cXJA,ABHYAAS EDUCATION,167000,5169057,1711,q4Q4dg6JDE0,प्रश्नावली 5.3 Class 10 Maths | NCERT Class 10...,2026-07-22T01:57:46Z,3985.0,520,23,2,📚 Class 10 Maths Chapter 5 – Arithmetic Progre...,https://www.youtube.com/watch?v=q4Q4dg6JDE0
3,Education,UCBg_mociSFb4ECbu91_cXJA,ABHYAAS EDUCATION,167000,5169057,1711,UTrN2vvjau8,Class 10 Maths | निर्देशांक ज्यामिति (Coordina...,2026-07-23T01:56:33Z,4655.0,566,27,0,Class 10 Maths | Coordinate Geometry Part 01 |...,https://www.youtube.com/watch?v=UTrN2vvjau8
4,Education,UCBg_mociSFb4ECbu91_cXJA,ABHYAAS EDUCATION,167000,5169057,1711,22S8ptzHl-g,Class 10 Maths | निर्देशांक ज्यामिति (Coordina...,2026-07-24T02:38:43Z,2185.0,355,23,3,Class 10 Maths | Coordinate Geometry Part 01 |...,https://www.youtube.com/watch?v=22S8ptzHl-g


In [4]:
required_columns=['category','subscriber_count','channel_view_count','duration_seconds', 'view_count']
df1=df[required_columns]
df1.head()

,category,subscriber_count,channel_view_count,duration_seconds,view_count
0,Education,167000,5169057,3988.0,427
1,Education,167000,5169057,3606.0,528
2,Education,167000,5169057,3985.0,520
3,Education,167000,5169057,4655.0,566
4,Education,167000,5169057,2185.0,355


In [5]:
from src.utils.remove_null_values import remove_null
df2=remove_null(df1)

X=df2.drop(columns=["view_count"])
y= np.log1p(df2["view_count"])

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

Total null values removed:  6
percent of null values removed:  0.06027122049221497


In [6]:
df2.head()

,category,subscriber_count,channel_view_count,duration_seconds,view_count
0,Education,167000,5169057,3988.0,427
1,Education,167000,5169057,3606.0,528
2,Education,167000,5169057,3985.0,520
3,Education,167000,5169057,4655.0,566
4,Education,167000,5169057,2185.0,355


In [7]:
from src.utils.combined_pipeline import combined_transform,model_with_combined_processor
numeric_features = ['subscriber_count','channel_view_count','duration_seconds']
categorical_features = ["category"]
combined_processor_pipeline= combined_transform(numeric_features,categorical_features)
model_pipeline = model_with_combined_processor(combined_processor_pipeline,RandomForestRegressor(n_estimators=500,random_state=42,n_jobs=-1))

In [8]:

from sklearn.metrics import r2_score, mean_squared_error

model_pipeline.fit(X_train, y_train)

predictions = model_pipeline.predict(X_test)

y_pred= model_pipeline.predict(X_test)
r2= r2_score(y_test,y_pred)
r2

0.7814785376886162

In [41]:
from src.utils.yaml_loader import yaml_loader
data= yaml_loader("./params.yaml")
data

{'params': {'model__n_estimators': [100, 200, 500, 700, 1000],
  'model__max_depth': ['None', 10, 20, 25, 30],
  'model__min_samples_split': [2, 5, 7, 10]}}

In [42]:
params= data["params"]
params

{'model__n_estimators': [100, 200, 500, 700, 1000],
 'model__max_depth': ['None', 10, 20, 25, 30],
 'model__min_samples_split': [2, 5, 7, 10]}

## finding final model

In [ ]:
import numpy as np 
import pandas as pd 
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import mlflow
import mlflow.sklearn
import dagshub

from src.utils.combined_pipeline import combined_transform,model_with_combined_processor
from src.utils.yaml_loader import yaml_loader

data= yaml_loader("./params.yaml")
params= data["params"]

results=[]

dagshub.init(repo_owner='AIforeverything', repo_name='MLOPS_youtube_CTA_Project', mlflow=True)

with mlflow.start_run(run_name='EXP-3_RandomForestRegressor') as run:
    
    numeric_features = ['subscriber_count','channel_view_count','duration_seconds']
    categorical_features = ["category"]
    combined_processor_pipeline= combined_transform(numeric_features,categorical_features)
    model_pipeline = model_with_combined_processor(combined_processor_pipeline,RandomForestRegressor(random_state=42,n_jobs=-1))
    
    grid_search= GridSearchCV(
        estimator= model_pipeline,
        param_grid=params,
        cv=5,
        verbose= True,
        scoring='neg_mean_squared_error',
        return_train_score= True 
    )
        
    mlflow.log_param('algorithm', 'RandomForestRegressor')
    mlflow.log_param('cv_fold',5)
    mlflow.log_param(
                "scoring",
                "neg_mean_squared_error"
            )

    # --------------------------------------------------
    # Train GridSearchCV
    # -------------------------------------------------
    grid_search.fit(
        X_train,
        y_train)
    
    # --------------------------------------------------
    # Best model
    # -------------------------------------------------
    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_
    best_cv_score = grid_search.best_score_
    # --------------------------------------------------
    # Log best parameters
    # -------------------------------------------------
    mlflow.log_params(best_params)
    mlflow.log_metric(
        "best_cv_mse",
        -best_cv_score)
    
    # --------------------------------------------------
    # Test prediction
    # -------------------------------------------------
    y_pred = best_model.predict(X_test)
    # --------------------------------------------------
    # Test metrics
    # -------------------------------------------------
    mse = mean_squared_error(
        y_test,
        y_pred)
    
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(
        y_test,
        y_pred)
    
    r2 = r2_score(
        y_test,
        y_pred)
    
    # --------------------------------------------------
    # Log test metrics
    # -------------------------------------------------
    mlflow.log_metric(
        "test_mse",
        mse)
    
    mlflow.log_metric(
        "test_rmse",
        rmse)
    
    mlflow.log_metric(
        "test_mae",
        mae)
    
    mlflow.log_metric(
        "test_r2",
        r2)
    
    # --------------------------------------------------
    # Log best model
    # -------------------------------------------------
    mlflow.sklearn.log_model(
        best_model,
        name="model",
        skops_trusted_types=["numpy.dtype",'xgboost.core.Booster', 'xgboost.sklearn.XGBRegressor'])
    
    # --------------------------------------------------
    # Store result
    # -------------------------------------------------
    results.append({
        "model": 'RandomForestRegressor',
        "best_params": best_params,
        "cv_mse": -best_cv_score,
        "test_mse": mse,
        "test_rmse": rmse,
        "test_mae": mae,
        "test_r2": r2,
    })
    print("Best Parameters:")
    print(best_params)
    print(f"CV MSE  : {-best_cv_score:.4f}")
    print(f"Test MSE: {mse:.4f}")
    print(f"Test RMSE: {rmse:.4f}")
    print(f"Test MAE : {mae:.4f}")
    print(f"Test R²  : {r2:.4f}")

Initialized MLflow to track repo "AIforeverything/MLOPS_youtube_CTA_Project"

Repository AIforeverything/MLOPS_youtube_CTA_Project initialized!

Fitting 5 folds for each of 100 candidates, totalling 500 fits


/home/om/Desktop/myenv/lib/python3.12/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
100 fits failed out of a total of 500.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
100 fits failed with the following error:
Traceback (most recent call last):
  File "/home/om/Desktop/myenv/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/om/Desktop/myenv/lib/python3.12/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/om/Desktop/myenv/lib/python3.12/site-packages/sklearn/pipeline.py", line 66

Best Parameters:
{'model__max_depth': 30, 'model__min_samples_split': 10, 'model__n_estimators': 1000}
CV MSE  : 1.4183
Test MSE: 1.2356
Test RMSE: 1.1116
Test MAE : 0.7796
Test R²  : 0.8115
🏃 View run EXP-3_RandomForestRegressor at: https://dagshub.com/AIforeverything/MLOPS_youtube_CTA_Project.mlflow/#/experiments/0/runs/a0e6bab737a74fa4a11e47463f90f56e
🧪 View experiment at: https://dagshub.com/AIforeverything/MLOPS_youtube_CTA_Project.mlflow/#/experiments/0


In [44]:
{'model__max_depth': 30, 'model__min_samples_split': 10, 'model__n_estimators': 1000}

{'model__max_depth': 30,
 'model__min_samples_split': 10,
 'model__n_estimators': 1000}